In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Standard imports
import numpy as np
import xarray as xr
import tqdm as tqdm

# For variogram estimation
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import label
from scipy.spatial.distance import pdist
from scipy.optimize import curve_fit

In [3]:
# Import OpenSense modules as submodules
import sys
import os

sys.path.append(os.path.abspath("./pycomlink/"))
sys.path.append(os.path.abspath("./poligrain/src/"))
sys.path.append(os.path.abspath("./mergeplg/src/"))

import pycomlink as pycml 
import poligrain as plg
import mergeplg 

In [4]:
os.makedirs('data/adjusted_fields', exist_ok=True)

In [5]:
# Define function to estimate variogram from rainfall event
def get_event_variogram(da_event, bin_edges, min_obs, plot_variogram=False):
    """
    da_event: xarray data for computing variogram
    bin_edges: array of distance bins [m] (e.g., np.linspace(0, 30000, 1500))
    min_obs: minimum observations needed to perform variogram estimation
    plot_variogram: whether to plot the variogram
    """
    all_distances = []
    all_sq_diffs = []
    
    # 1. Collect pairs from all time steps in the event
    n_obs = 0
    for t in da_event.time:
        # Extract data for this timestamp and drop nan
        data_t = da_event.sel(time=t).dropna(dim='cml_id')
        
        # We need at least 2 points to make a pair
        if len(data_t.cml_id) < 2:
            continue

        n_obs += data_t.cml_id.size

        # Get coordinates of data
        coords = np.column_stack([data_t.x, data_t.y])
        values = data_t.values
        
        # Calculate distances and 0.5 * (zi - zj)^2
        dist = pdist(coords, metric='euclidean')

        # Calculate squared distance (variance)
        sq_diff = 0.5 * pdist(values[:, None], 'sqeuclidean')
        
        all_distances.append(dist)
        all_sq_diffs.append(sq_diff)

    # If not enough observations
    if n_obs < min_obs:
        return None
        
    # Flatten into two long arrays of all pairs in the event
    dist_pool = np.concatenate(all_distances)
    diff_pool = np.concatenate(all_sq_diffs)
    
    # 2. Binning 
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    gamma_obs = []
    count = []

    # Also drop largest outliers
    #diff_upper = np.nanquantile(dist_pool, q = 0.95)
    
    for i in range(len(bin_edges)-1):
        mask = (dist_pool > bin_edges[i]) & (dist_pool <= bin_edges[i+1]) # & (diff_pool <= diff_upper) 
        if np.any(mask):
            gamma_obs.append(np.mean(diff_pool[mask]))
            count.append(diff_pool[mask].size)
        else:
            gamma_obs.append(np.nan)
            count.append(0)

    count = np.array(count)
    gamma_obs = np.array(gamma_obs)

    # 3. Define valid observation bins and get variance
    valid = ~np.isnan(gamma_obs) & (count != 0) 

    # Locate where most observations are
    dist_lower = np.nanquantile(dist_pool, q = 0.05)
    dist_upper = np.nanquantile(dist_pool, q = 0.95)
    ind_lower = np.where(dist_lower < bin_centers[valid])[0][0]

    if (dist_upper > bin_centers[valid]).all():
        ind_upper = bin_centers[valid].size
    else:
        ind_upper = np.where(dist_upper < bin_centers[valid])[0][0]

    total_variance = np.nanmean(gamma_obs[valid][ind_lower:ind_upper])
    gamma_obs_norm = gamma_obs/total_variance

    # If no variance 
    if total_variance == 0: 
        return None
        
    # 4. Define variogram and bounds
    def spherical_model(h, nugget, p_sill, range_a):
        # Use same definitions as pykrige:
        # https://geostat-framework.readthedocs.io/projects/pykrige/en/stable/variogram_models.html
        return np.where(h <= range_a, 
                        p_sill * (1.5 * (h/range_a) - 0.5 * (h/range_a)**3) + nugget, 
                        p_sill + nugget)
        
    # Estimate partial sill from normalized variance
    def f(h, r):
        return spherical_model(h, 0.2, 0.8, r)
        
    # Initial guess: [min(gamma), max_dist/2]
    p0 = [np.max(bin_centers[valid][ind_lower:ind_upper])/2]

    # Parameter bounds
    bound_l_range = bin_centers[valid][ind_lower]
    bound_u_range = np.max(bin_edges)*2
    
    bounds = [bound_l_range, bound_u_range]
    
    # 5. Optimize and return parameters
    popt, _ = curve_fit(
        f, 
        bin_centers[valid][:ind_upper], 
        gamma_obs_norm[valid][:ind_upper], 
        p0=p0, 
        bounds=bounds,
    )

    nugget = 0.2
    p_sill = 1 - nugget
    range_a = popt[0]

    if plot_variogram:
        fig, ax = plt.subplots(1, 1)
        ax.plot(bin_centers[valid], spherical_model(bin_centers[valid], nugget, p_sill, range_a), label='plot variogram')
        ax.plot(bin_centers[valid][:ind_upper], gamma_obs_norm[valid][:ind_upper], label='data used')
        ax.plot(bin_centers[valid][ind_upper:], gamma_obs_norm[valid][ind_upper:], label='data left out')
        plt.legend()
        plt.show()
    
    return n_obs, [nugget, p_sill, range_a]


# OpenMRG adjustment

In [6]:
# OpenMRG
ds_rad = xr.open_dataset("data/andersson_2022_OpenMRG/radar/openmrg_rad.nc")                    
ds_cmls = xr.open_dataset("data/processed_cml_OpenMRG.nc")       

# Get radar along CML, used for variogram estimation
da_intersect_weights = plg.spatial.calc_sparse_intersect_weights_for_several_cmls(
    x1_line=ds_cmls.site_0_lon.values,
    y1_line=ds_cmls.site_0_lat.values,
    x2_line=ds_cmls.site_1_lon.values,
    y2_line=ds_cmls.site_1_lat.values,
    cml_id=ds_cmls.cml_id.values,
    x_grid=ds_rad.lon.values,
    y_grid=ds_rad.lat.values,
    grid_point_location='center',
)
ds_cmls['radar_along_cml'] = plg.spatial.get_grid_time_series_at_intersections(
    grid_data=ds_rad.rainfall_amount,
    intersect_weights=da_intersect_weights,
)

# Difference used for additive
ds_cmls['rainfall_difference'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc - ds_cmls.radar_along_cml, 
    np.nan
)

# Ratio used for multiplicative
ds_cmls['rainfall_ratio'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc/ds_cmls.radar_along_cml, 
    np.nan
)

# CML obs used for KED
ds_cmls['rainfall_cml'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc, 
    np.nan
)

months = ['2015-06', '2015-07', '2015-08']                           

In [7]:
# methods parameter : default version
nnear = 70 
diff_check_sel = 5
ratio_check_sel = (0.2,8)

In [8]:
# additive IDW conservative check
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # IDW merging initialization 
    merger = mergeplg.merge.MergeDifferenceIDW(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        method="additive",
        nnear=nnear,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"add_p_idw": xr.concat(rainfall, dim="time") })
data.to_netcdf('data/adjusted_fields/OpenMRG_add_p_idw_cc.nc')    
del data, merger

month: 2015-06


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 719/719 [00:13<00:00, 54.36it/s]


month: 2015-07


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [00:13<00:00, 56.00it/s]


month: 2015-08


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [00:13<00:00, 56.67it/s]


In [9]:
# additive IDW no check
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # IDW merging initialization 
    merger = mergeplg.merge.MergeDifferenceIDW(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        method="additive",
        nnear=nnear,
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"add_p_idw": xr.concat(rainfall, dim="time") })
data.to_netcdf('data/adjusted_fields/OpenMRG_add_p_idw_nc.nc')    
del data, merger

month: 2015-06


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 719/719 [00:13<00:00, 53.91it/s]


month: 2015-07


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [00:13<00:00, 53.22it/s]


month: 2015-08


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [00:14<00:00, 52.98it/s]


In [10]:
# multiplicative IDW conservative check
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # IDW merging initialization 
    merger = mergeplg.merge.MergeDifferenceIDW(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        method="multiplicative",
        nnear=nnear,
        range_checks={'ratio_check':ratio_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"mul_p_idw": xr.concat(rainfall, dim="time") })
data.to_netcdf('data/adjusted_fields/OpenMRG_mul_p_idw_cc.nc')    
del data, merger

month: 2015-06


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 719/719 [00:13<00:00, 51.64it/s]


month: 2015-07


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [00:14<00:00, 52.33it/s]


month: 2015-08


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [00:14<00:00, 52.01it/s]


In [11]:
# multiplicative IDW no check
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # IDW merging initialization 
    merger = mergeplg.merge.MergeDifferenceIDW(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        method="multiplicative",
        nnear=nnear,
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"mul_p_idw": xr.concat(rainfall, dim="time") })
data.to_netcdf('data/adjusted_fields/OpenMRG_mul_p_idw_nc.nc')    
del data, merger

month: 2015-06


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 719/719 [00:13<00:00, 51.56it/s]


month: 2015-07


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [00:14<00:00, 49.98it/s]


month: 2015-08


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [00:14<00:00, 49.95it/s]


In [12]:
# additive POINT ORDINARY KRIGING conservative check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_difference, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)
data = xr.Dataset({"add_p_ok": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_add_p_ok_cc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:21<00:00,  7.83it/s]


In [13]:
# additive POINT ORDINARY KRIGING no check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_difference, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)
data = xr.Dataset({"add_p_ok": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_add_p_ok_nc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:26<00:00,  6.27it/s]


In [14]:
# multiplicative POINT ORDINARY KRIGING conservative check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_ratio, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"mul_p_ok": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_mul_p_ok_cc.nc')    
del data, merger

 90%|███████████████████████████████████████████████████████████████████████████████████████████████████████            | 26/29 [00:12<00:00,  3.17it/s]/tmp/ipykernel_51920/2791475567.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:15<00:00, 10.84it/s]


In [15]:
# multiplicative POINT ORDINARY KRIGING no check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_ratio, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"mul_p_ok": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_mul_p_ok_nc.nc')    
del data, merger

 90%|███████████████████████████████████████████████████████████████████████████████████████████████████████            | 26/29 [00:09<00:00,  3.41it/s]/tmp/ipykernel_51920/2791475567.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:21<00:00,  7.96it/s]


In [16]:
# additive BLOCK ORDINARY KRIGING conservative check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_difference, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"add_b_ok": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_add_b_ok_cc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:22<00:00,  7.40it/s]


In [17]:
# additive BLOCK ORDINARY KRIGING no check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_difference, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"add_b_ok": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_add_b_ok_nc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:23<00:00,  7.26it/s]


In [18]:
# multiplicative BLOCK ORDINARY KRIGING conservative check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_ratio, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"mul_b_ok": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_mul_b_ok_cc.nc')    
del data, merger

 90%|███████████████████████████████████████████████████████████████████████████████████████████████████████            | 26/29 [00:11<00:00,  3.01it/s]/tmp/ipykernel_51920/2791475567.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:16<00:00, 10.45it/s]


In [19]:
# multiplicative BLOCK ORDINARY KRIGING no check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_ratio, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"mul_b_ok": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_mul_b_ok_nc.nc')    
del data, merger

 90%|███████████████████████████████████████████████████████████████████████████████████████████████████████            | 26/29 [00:10<00:00,  3.29it/s]/tmp/ipykernel_51920/2791475567.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:21<00:00,  7.81it/s]


In [20]:
# KED point conservative check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_cml, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"ked_p": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_ked_p_cc.nc')    
del data, merger

 90%|███████████████████████████████████████████████████████████████████████████████████████████████████████            | 26/29 [00:10<00:00,  3.28it/s]/tmp/ipykernel_51920/2791475567.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:23<00:00,  7.30it/s]


In [21]:
# KED point no check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_cml, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=False,
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"ked_p": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_ked_p_nc.nc')    
del data, merger

 90%|███████████████████████████████████████████████████████████████████████████████████████████████████████            | 26/29 [00:10<00:00,  3.34it/s]/tmp/ipykernel_51920/2791475567.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:23<00:00,  7.27it/s]


In [22]:
# KED block conservative check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_cml, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"ked_b": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_ked_b_cc.nc')    
del data, merger

 90%|███████████████████████████████████████████████████████████████████████████████████████████████████████            | 26/29 [00:10<00:00,  3.20it/s]/tmp/ipykernel_51920/2791475567.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:22<00:00,  7.35it/s]


In [23]:
# KED block no check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_cml, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=True,
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"ked_b": data})
data.to_netcdf('data/adjusted_fields/OpenMRG_ked_b_nc.nc')    
del data, merger

 90%|███████████████████████████████████████████████████████████████████████████████████████████████████████            | 26/29 [00:10<00:00,  3.30it/s]/tmp/ipykernel_51920/2791475567.py:79: RuntimeWarning: invalid value encountered in divide
  gamma_obs_norm = gamma_obs/total_variance
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 168/168 [00:23<00:00,  7.24it/s]


In [24]:
dataset = 'OpenMRG'
adj_path = 'data/adjusted_fields/'

ds_gauges = xr.open_dataset('data/andersson_2022_OpenMRG/gauges/openmrg_gauges.nc')     
ds_rad = xr.open_dataset("data/andersson_2022_OpenMRG/radar/openmrg_rad.nc")          
ds_cmls = xr.open_dataset("data/processed_cml_OpenMRG.nc").R_acc            

# adding adjusted field and naming
ds_rad = ds_rad.rename({'rainfall_amount':'radar'})  
ds_rad['add_p_idw']=xr.open_dataset(adj_path+dataset+'_add_p_idw_cc.nc').add_p_idw
ds_rad['add_p_ok']=xr.open_dataset(adj_path+dataset+'_add_p_ok_cc.nc').add_p_ok
ds_rad['add_b_ok']=xr.open_dataset(adj_path+dataset+'_add_b_ok_cc.nc').add_b_ok
ds_rad['mul_p_idw']=xr.open_dataset(adj_path+dataset+'_mul_p_idw_cc.nc').mul_p_idw
ds_rad['mul_p_ok']=xr.open_dataset(adj_path+dataset+'_mul_p_ok_cc.nc').mul_p_ok
ds_rad['mul_b_ok']=xr.open_dataset(adj_path+dataset+'_mul_b_ok_cc.nc').mul_b_ok
ds_rad['ked_p']=xr.open_dataset(adj_path+dataset+'_ked_p_cc.nc').ked_p
ds_rad['ked_b']=xr.open_dataset(adj_path+dataset+'_ked_b_cc.nc').ked_b

fields = ['radar', 'add_p_idw', 'add_p_ok', 
          'add_b_ok', 'ked_p', 'ked_b',
          'mul_p_idw', 'mul_p_ok', 'mul_b_ok']

# RAINFALL FIELDS AT THE RAIN GAUGES
get_grid_at_points = plg.spatial.GridAtPoints(
    da_gridded_data=ds_rad.isel(time = 0), 
    da_point_data=ds_gauges.isel(time = 0),
    nnear=1,
    stat="best" 
)

for field in fields:
    ds_gauges[field] = get_grid_at_points(
        da_gridded_data=ds_rad[field],
        da_point_data=ds_gauges.rainfall_amount,  
    )

# COMPUTE METRICS
threshold = 0.2

for field in fields:
        if field == 'radar':
            metric = pd.DataFrame([plg.validation.calculate_rainfall_metrics(
                reference=ds_gauges.rainfall_amount.values.flatten(),
                estimate=ds_gauges[field].values.flatten(),
                ref_thresh=threshold,
                est_thresh=threshold,
            )]) 
            metric['dataset'] = 'OpenMRG'
            metric['method'] = field
            result = metric
        else:
            metric = pd.DataFrame([plg.validation.calculate_rainfall_metrics(
                reference=ds_gauges.rainfall_amount.values.flatten(),
                estimate=ds_gauges[field].values.flatten(),
                ref_thresh=threshold,
                est_thresh=threshold,
            )]) 
            metric['dataset'] = 'OpenMRG'
            metric['method'] = field
            result = pd.concat([result, metric])

result.to_csv('metrics_OpenMRG_cc.csv')

In [25]:
dataset = 'OpenMRG'
adj_path = 'data/adjusted_fields/'

ds_gauges = xr.open_dataset('data/andersson_2022_OpenMRG/gauges/openmrg_gauges.nc')     
ds_rad = xr.open_dataset("data/andersson_2022_OpenMRG/radar/openmrg_rad.nc")          
ds_cmls = xr.open_dataset("data/processed_cml_OpenMRG.nc").R_acc        

# adding adjusted field and naming
ds_rad = ds_rad.rename({'rainfall_amount':'radar'})  
ds_rad['add_p_idw']=xr.open_dataset(adj_path+dataset+'_add_p_idw_nc.nc').add_p_idw
ds_rad['add_p_ok']=xr.open_dataset(adj_path+dataset+'_add_p_ok_nc.nc').add_p_ok
ds_rad['add_b_ok']=xr.open_dataset(adj_path+dataset+'_add_b_ok_nc.nc').add_b_ok
ds_rad['mul_p_idw']=xr.open_dataset(adj_path+dataset+'_mul_p_idw_nc.nc').mul_p_idw
ds_rad['mul_p_ok']=xr.open_dataset(adj_path+dataset+'_mul_p_ok_nc.nc').mul_p_ok
ds_rad['mul_b_ok']=xr.open_dataset(adj_path+dataset+'_mul_b_ok_nc.nc').mul_b_ok
ds_rad['ked_p']=xr.open_dataset(adj_path+dataset+'_ked_p_nc.nc').ked_p
ds_rad['ked_b']=xr.open_dataset(adj_path+dataset+'_ked_b_nc.nc').ked_b

fields = ['radar', 'add_p_idw', 'add_p_ok', 
          'add_b_ok', 'ked_p', 'ked_b',
          'mul_p_idw', 'mul_p_ok', 'mul_b_ok']

# RAINFALL FIELDS AT THE RAIN GAUGES
get_grid_at_points = plg.spatial.GridAtPoints(
    da_gridded_data=ds_rad.isel(time = 0), 
    da_point_data=ds_gauges.isel(time = 0),
    nnear=1,
    stat="best" 
)

for field in fields:
    ds_gauges[field] = get_grid_at_points(
        da_gridded_data=ds_rad[field],
        da_point_data=ds_gauges.rainfall_amount,  
    )

# COMPUTE METRICS
threshold = 0.2

for field in fields:
        if field == 'radar':
            metric = pd.DataFrame([plg.validation.calculate_rainfall_metrics(
                reference=ds_gauges.rainfall_amount.values.flatten(),
                estimate=ds_gauges[field].values.flatten(),
                ref_thresh=threshold,
                est_thresh=threshold,
            )]) 
            metric['dataset'] = 'OpenMRG'
            metric['method'] = field
            result = metric
        else:
            metric = pd.DataFrame([plg.validation.calculate_rainfall_metrics(
                reference=ds_gauges.rainfall_amount.values.flatten(),
                estimate=ds_gauges[field].values.flatten(),
                ref_thresh=threshold,
                est_thresh=threshold,
            )]) 
            metric['dataset'] = 'OpenMRG'
            metric['method'] = field
            result = pd.concat([result, metric])

result.to_csv('metrics_OpenMRG_nc.csv')

# OpenRainER adjustment

In [26]:
# OpenRainER
ds_rad = xr.open_dataset("data/covi_2024_OpenRainER/openrainer_radar.nc")         
ds_cmls = xr.open_dataset("data/processed_cml_OpenRainER.nc")   
ds_gauges = xr.open_dataset('data/covi_2024_OpenRainER/AWS_rainfall.nc')        

# Get radar along CML, used for variogram estimation
da_intersect_weights = plg.spatial.calc_sparse_intersect_weights_for_several_cmls(
    x1_line=ds_cmls.site_0_lon.values,
    y1_line=ds_cmls.site_0_lat.values,
    x2_line=ds_cmls.site_1_lon.values,
    y2_line=ds_cmls.site_1_lat.values,
    cml_id=ds_cmls.cml_id.values,
    x_grid=ds_rad.lon.values,
    y_grid=ds_rad.lat.values,
    grid_point_location='center',
)
ds_cmls['radar_along_cml'] = plg.spatial.get_grid_time_series_at_intersections(
    grid_data=ds_rad.rainfall_amount,
    intersect_weights=da_intersect_weights,
)

# Difference used for additive
ds_cmls['rainfall_difference'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc - ds_cmls.radar_along_cml, 
    np.nan
)

# Ratio used for multiplicative
ds_cmls['rainfall_ratio'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc/ds_cmls.radar_along_cml, 
    np.nan
)

# CML obs used for KED
ds_cmls['rainfall_cml'] = xr.where(
    ds_cmls.radar_along_cml > 0, 
    ds_cmls.R_acc, 
    np.nan
)   

months = ['2022-06', '2022-07', '2022-08']

In [27]:
# methods parameter : default version
nnear = 70 
diff_check_sel = 5
ratio_check_sel = (0.2,8)

In [28]:
# additive IDW conservative check
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # IDW merging initialization 
    merger = mergeplg.merge.MergeDifferenceIDW(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        method="additive",
        nnear=nnear,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.sel(time=time).R_acc,
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"add_p_idw": xr.concat(rainfall, dim="time") })
data.to_netcdf('data/adjusted_fields/OpenRainER_add_p_idw_cc.nc')    
del data, merger

month: 2022-06


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 719/719 [01:59<00:00,  6.01it/s]


month: 2022-07


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [02:03<00:00,  6.03it/s]


month: 2022-08


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [01:59<00:00,  6.20it/s]


In [29]:
# additive IDW no check
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # IDW merging initialization 
    merger = mergeplg.merge.MergeDifferenceIDW(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        method="additive",
        nnear=nnear,
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.sel(time=time).R_acc,
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"add_p_idw": xr.concat(rainfall, dim="time") })
data.to_netcdf('data/adjusted_fields/OpenRainER_add_p_idw_nc.nc')    
del data, merger

month: 2022-06


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 719/719 [01:57<00:00,  6.10it/s]


month: 2022-07


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [01:54<00:00,  6.49it/s]


month: 2022-08


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [01:59<00:00,  6.25it/s]


In [30]:
# multiplicative IDW conservative check
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # IDW merging initialization 
    merger = mergeplg.merge.MergeDifferenceIDW(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        method="multiplicative",
        nnear=nnear,
        range_checks={'ratio_check':ratio_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.sel(time=time).R_acc,
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"mul_p_idw": xr.concat(rainfall, dim="time") })
data.to_netcdf('data/adjusted_fields/OpenRainER_mul_p_idw_cc.nc')    
del data, merger

month: 2022-06


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 719/719 [01:51<00:00,  6.45it/s]


month: 2022-07


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [01:54<00:00,  6.50it/s]


month: 2022-08


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [01:56<00:00,  6.37it/s]


In [31]:
# multiplicative IDW no check
rainfall = []

for month in months:
    print('month: '+month)
    msel_ds_rad = ds_rad.sel(time = month) 
    msel_ds_cmls = ds_cmls.sel(time = month) 
    # IDW merging initialization 
    merger = mergeplg.merge.MergeDifferenceIDW(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        method="multiplicative",
        nnear=nnear,
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.sel(time=time).R_acc,
            )
        )
# saving the field + removing NaNs 
data = xr.Dataset({"mul_p_idw": xr.concat(rainfall, dim="time") })
data.to_netcdf('data/adjusted_fields/OpenRainER_mul_p_idw_nc.nc')    
del data, merger

month: 2022-06


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 719/719 [01:55<00:00,  6.22it/s]


month: 2022-07


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [01:55<00:00,  6.45it/s]


month: 2022-08


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [02:10<00:00,  5.69it/s]


In [32]:
# additive POINT ORDINARY KRIGING conservative check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_difference, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"add_p_ok": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_add_p_ok_cc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:45<00:00,  1.60it/s]


In [33]:
# additive POINT ORDINARY KRIGING no check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_difference, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"add_p_ok": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_add_p_ok_nc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:54<00:00,  1.51it/s]


In [34]:
# multiplicative POINT ORDINARY KRIGING conservative check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_ratio, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"mul_p_ok": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_mul_p_ok_cc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [01:34<00:00,  2.80it/s]


In [35]:
# multiplicative POINT ORDINARY KRIGING no check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_ratio, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=False,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"mul_p_ok": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_mul_p_ok_nc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:53<00:00,  1.52it/s]


In [36]:
# additive BLOCK ORDINARY KRIGING conservative check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_difference, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"add_b_ok": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_add_b_ok_cc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:50<00:00,  1.55it/s]


In [37]:
# additive BLOCK ORDINARY KRIGING no check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_difference, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="additive",
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"add_b_ok": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_add_b_ok_nc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:50<00:00,  1.55it/s]


In [38]:
# multiplicative BLOCK ORDINARY KRIGING conservative check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_ratio, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={'ratio_check':ratio_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"mul_b_ok": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_mul_b_ok_cc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [01:33<00:00,  2.82it/s]


In [39]:
# multiplicative BLOCK ORDINARY KRIGING no check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_ratio, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeDifferenceOrdinaryKriging(
        ds_rad=msel_ds_rad.rainfall_amount,
        ds_cmls=msel_ds_cmls,
        full_line=True,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        method="multiplicative",
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"mul_b_ok": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_mul_b_ok_nc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:57<00:00,  1.49it/s]


In [40]:
# KED point conservative check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_cml, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=False,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"ked_p": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_ked_p_cc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:45<00:00,  1.60it/s]


In [41]:
# KED point no check
# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_cml, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=False,
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"ked_p": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_ked_p_nc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:52<00:00,  1.53it/s]


In [42]:
# KED block conservative check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_cml, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=True,
        range_checks={'diff_check':diff_check_sel},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"ked_b": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_ked_b_cc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:47<00:00,  1.58it/s]


In [43]:
# KED block no check

# Variogram difference radar-cml
bin_edges = np.arange(0, 60000, 1000) # Varigroam bin size
min_obs = 100 # Minimum observations required to perform variogram estimation, else expand search

# Group nearby rainfall (closer than 6 hours) to estimate variogram
mask = (ds_cmls.radar_along_cml > 0.01).any(dim='cml_id').rolling(time=12, center=True).sum() > 0
mask = mask.fillna(0).astype(int) # Ensure no NaNs and int type for labeling
labels_raw, num_features = label(mask)
labels = xr.DataArray(labels_raw, coords=mask.coords, dims=mask.dims)

variograms = []
# Estimate variogram for rainfall events
for i in tqdm.tqdm(range(1, num_features + 1)):
    # neighbouring events to include
    n_neighbors = 0 

    # Event for variogram estimation
    target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
    combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)

    # Expand neighbourhood until we have enough data
    expand = True
    while expand:
        # Get start-end time
        time_start = combined_event_time.time.values[0]
        time_end = combined_event_time.time.values[-1]
        time_mid = combined_event_time.time.values[int(combined_event_time.time.size/2)]
        
        # Estimate variogram 
        out = get_event_variogram(
            ds_cmls.sel(time = slice(time_start, time_end)).rainfall_cml, # 
            bin_edges, 
            min_obs,
            plot_variogram = False,
        )

        # If variogram estimation was successful, stop expanding
        if out is not None:
            expand = False 
            n_obs = out[0]
            variogram_parameters = {"nugget": out[1][0], "sill": out[1][1], "range": out[1][2]}
            variograms.append([time_mid, out[1][0], out[1][1], out[1][2]])
            
        else:
            # Timesteps included in event
            n_time = combined_event_time.time.size

            # Expand neighborhood
            n_neighbors +=1
            target_labels = np.arange(max(1, i - n_neighbors), min(num_features, i + n_neighbors) + 1)
            combined_event_time = mask.time.where(labels.isin(target_labels), drop=True)
    
            # Break loop if exand did not result in more timesteps
            if n_time >= combined_event_time.time.size:
                expand = False
                print('Warning: Not enough rainfall obs in time series, using standard instead')
                variogram_parameters = {"nugget": 0.2, "sill": 0.8, "range": 30000}

variograms = pd.DataFrame(variograms, columns=['time', 'nugget', 'p_sill', 'range']).set_index('time')

# Aggregate variogram parameters across timesteps
variograms = variograms.resample('14D', label='left').mean()

# Add first and last timestep of CML to variogram dataframe, completing the time series
start_row = pd.DataFrame([variograms.iloc[0].values], index=[ds_cmls.time[0].values], columns=variograms.columns)
end_row = pd.DataFrame([variograms.iloc[-1].values], index=[ds_cmls.time[-1].values], columns=variograms.columns)
variograms = pd.concat([start_row, variograms, end_row]).sort_index()

# Merge rainfall events for aggregated variograms
rainfall = []
ref_times = variograms.sort_index().index 
for i in range(len(ref_times)-1):
    time_start = ref_times[i]
    time_end = ref_times[i +1]
    
    msel_ds_rad = ds_rad.sel(time = slice(time_start, time_end)) 
    msel_ds_cmls = ds_cmls.sel(time = slice(time_start, time_end)) 
    
    # OK merging initialization
    merger = mergeplg.merge.MergeKrigingExternalDrift(
        ds_rad=ds_rad.rainfall_amount,
        ds_cmls=ds_cmls,
        variogram_parameters=variogram_parameters,
        nnear=nnear,
        full_line=True,
        range_checks={},
    )
    # merging application (loop on timesteps)
    for time in tqdm.tqdm(msel_ds_rad.time.data):
        rainfall.append(
            merger(
                da_rad=msel_ds_rad.sel(time=time).rainfall_amount,
                da_cmls=msel_ds_cmls.R_acc.sel(time=time),
            )
        )
        
# Concat rainfall events
data = xr.concat(rainfall, dim="time")

# Overlapping timesteps in merging loop, remove them
data = data.drop_duplicates('time').reindex_like(ds_rad, method=None)

# Fill missing timesteps with radar, or 0 if radar is nan
data = data.fillna(ds_rad.rainfall_amount)
data = data.fillna(0)

# Save
data = xr.Dataset({"ked_b": data})
data.to_netcdf('data/adjusted_fields/OpenRainER_ked_b_nc.nc')    
del data, merger

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 264/264 [02:52<00:00,  1.53it/s]


In [44]:
dataset = 'OpenRainER'
adj_path = 'data/adjusted_fields/'

ds_gauges = xr.open_dataset('data/covi_2024_OpenRainER/AWS_rainfall.nc')        
ds_rad = xr.open_dataset("data/covi_2024_OpenRainER/openrainer_radar.nc")       
ds_cmls = xr.open_dataset("data/processed_cml_OpenRainER.nc").R_acc             

# adding adjusted field and naming
ds_rad = ds_rad.rename({'rainfall_amount':'radar'})  
ds_rad['add_p_idw']=xr.open_dataset(adj_path+dataset+'_add_p_idw_cc.nc').add_p_idw
ds_rad['add_p_ok']=xr.open_dataset(adj_path+dataset+'_add_p_ok_cc.nc').add_p_ok
ds_rad['add_b_ok']=xr.open_dataset(adj_path+dataset+'_add_b_ok_cc.nc').add_b_ok
ds_rad['mul_p_idw']=xr.open_dataset(adj_path+dataset+'_mul_p_idw_cc.nc').mul_p_idw
ds_rad['mul_p_ok']=xr.open_dataset(adj_path+dataset+'_mul_p_ok_cc.nc').mul_p_ok
ds_rad['mul_b_ok']=xr.open_dataset(adj_path+dataset+'_mul_b_ok_cc.nc').mul_b_ok
ds_rad['ked_p']=xr.open_dataset(adj_path+dataset+'_ked_p_cc.nc').ked_p
ds_rad['ked_b']=xr.open_dataset(adj_path+dataset+'_ked_b_cc.nc').ked_b

fields = ['radar', 'add_p_idw', 'add_p_ok', 
          'add_b_ok', 'ked_p', 'ked_b',
          'mul_p_idw', 'mul_p_ok', 'mul_b_ok']
# RAINFALL FIELDS AT THE RAIN GAUGES
get_grid_at_points = plg.spatial.GridAtPoints(
    da_gridded_data=ds_rad.isel(time = 0), 
    da_point_data=ds_gauges.isel(time = 0),
    nnear=1,
    stat="best" 
)

for field in fields:
    ds_gauges[field] = get_grid_at_points(
        da_gridded_data=ds_rad[field],
        da_point_data=ds_gauges.rainfall_amount,  
    )

# COMPUTE METRICS
threshold = 0.2

for field in fields:
        if field == 'radar':
            metric = pd.DataFrame([plg.validation.calculate_rainfall_metrics(
                reference=ds_gauges.rainfall_amount.values.flatten(),
                estimate=ds_gauges[field].values.flatten(),
                ref_thresh=threshold,
                est_thresh=threshold,
            )]) 
            metric['dataset'] = 'OpenRainER'
            metric['method'] = field
            result = metric
        else:
            metric = pd.DataFrame([plg.validation.calculate_rainfall_metrics(
                reference=ds_gauges.rainfall_amount.values.flatten(),
                estimate=ds_gauges[field].values.flatten(),
                ref_thresh=threshold,
                est_thresh=threshold,
            )]) 
            metric['dataset'] = 'OpenRainER'
            metric['method'] = field
            result = pd.concat([result, metric])

result.to_csv('metrics_OpenRainER_cc.csv')

In [45]:
dataset = 'OpenRainER'
adj_path = 'data/adjusted_fields/'

ds_gauges = xr.open_dataset('data/covi_2024_OpenRainER/AWS_rainfall.nc')        
ds_rad = xr.open_dataset("data/covi_2024_OpenRainER/openrainer_radar.nc")       
ds_cmls = xr.open_dataset("data/processed_cml_OpenRainER.nc").R_acc             

# adding adjusted field and naming
ds_rad = ds_rad.rename({'rainfall_amount':'radar'})  
ds_rad['add_p_idw']=xr.open_dataset(adj_path+dataset+'_add_p_idw_nc.nc').add_p_idw
ds_rad['add_p_ok']=xr.open_dataset(adj_path+dataset+'_add_p_ok_nc.nc').add_p_ok
ds_rad['add_b_ok']=xr.open_dataset(adj_path+dataset+'_add_b_ok_nc.nc').add_b_ok
ds_rad['mul_p_idw']=xr.open_dataset(adj_path+dataset+'_mul_p_idw_nc.nc').mul_p_idw
ds_rad['mul_p_ok']=xr.open_dataset(adj_path+dataset+'_mul_p_ok_nc.nc').mul_p_ok
ds_rad['mul_b_ok']=xr.open_dataset(adj_path+dataset+'_mul_b_ok_nc.nc').mul_b_ok
ds_rad['ked_p']=xr.open_dataset(adj_path+dataset+'_ked_p_nc.nc').ked_p
ds_rad['ked_b']=xr.open_dataset(adj_path+dataset+'_ked_b_nc.nc').ked_b

fields = ['radar', 'add_p_idw', 'add_p_ok', 
          'add_b_ok', 'ked_p', 'ked_b',
          'mul_p_idw', 'mul_p_ok', 'mul_b_ok']
# RAINFALL FIELDS AT THE RAIN GAUGES
get_grid_at_points = plg.spatial.GridAtPoints(
    da_gridded_data=ds_rad.isel(time = 0), 
    da_point_data=ds_gauges.isel(time = 0),
    nnear=1,
    stat="best" 
)

for field in fields:
    ds_gauges[field] = get_grid_at_points(
        da_gridded_data=ds_rad[field],
        da_point_data=ds_gauges.rainfall_amount,  
    )

# COMPUTE METRICS
threshold = 0.2

for field in fields:
        if field == 'radar':
            metric = pd.DataFrame([plg.validation.calculate_rainfall_metrics(
                reference=ds_gauges.rainfall_amount.values.flatten(),
                estimate=ds_gauges[field].values.flatten(),
                ref_thresh=threshold,
                est_thresh=threshold,
            )]) 
            metric['dataset'] = 'OpenRainER'
            metric['method'] = field
            result = metric
        else:
            metric = pd.DataFrame([plg.validation.calculate_rainfall_metrics(
                reference=ds_gauges.rainfall_amount.values.flatten(),
                estimate=ds_gauges[field].values.flatten(),
                ref_thresh=threshold,
                est_thresh=threshold,
            )]) 
            metric['dataset'] = 'OpenRainER'
            metric['method'] = field
            result = pd.concat([result, metric])

result.to_csv('metrics_OpenRainER_nc.csv')